In [2]:
import pandas as pd
import os
import re
import ast
import time
import getpass
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

print("1. Setting up API & LLM...")
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key (gsk_...): ")

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)

print("2. Connecting to the Massive Vector Database...")
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
persist_dir = "../data/chroma_db_massive"
vector_db = Chroma(collection_name="massive_physics_db", embedding_function=embedding_model, persist_directory=persist_dir)

print("3. Building the EXTREME Bulletproof XML Prompt...")
prompt_template = PromptTemplate(
    input_variables=["context", "question", "choices"],
    template="""You are an expert physics professor answering a multiple-choice exam.
Read the context, question, and choices carefully.
You MUST provide your final answer wrapped in XML tags like this: <answer>INDEX</answer>.

CRITICAL RULES:
1. Even if the context does not contain the exact answer, you MUST make an educated guess.
2. The INDEX must be a single digit (0, 1, 2, or 3).
3. Do NOT write any apologies, explanations, or extra text. ONLY the XML tag.

CONTEXT:
{context}

QUESTION: 
{question}

CHOICES:
{choices}

Provide your answer:"""
)

def extract_choices(choices_str):
    matches = re.findall(r"'([^']*)'", str(choices_str))
    if len(matches) != 4:
        matches = re.findall(r'"([^"]*)"', str(choices_str))
    return matches

print("4. Loading Benchmark Dataset (CSV)...")
csv_path = "../data/benchmark/physics_evaluation_benchmark.csv"

try:
    df = pd.read_csv(csv_path)
    evaluation_results = []
    
    print("="*50 + "\nSTARTING MASSIVE EVALUATION (RATE-LIMIT PROTECTED)\n" + "="*50)
    
    for index, row in df.iterrows():
        question_text = row['question']
        choices_list = extract_choices(row['choices'])
        choices_text = "\n".join([f"Index {i}: {choice}" for i, choice in enumerate(choices_list)])

        retrieved_docs = vector_db.similarity_search(question_text, k=3)
        context_text = "\n\n---\n\n".join([doc.page_content for doc in retrieved_docs])
        
        final_prompt = prompt_template.format(context=context_text, question=question_text, choices=choices_text)
        
        # --- RATE LIMIT PROTECTION LOOP ---
        max_retries = 5
        response_text = ""
        for attempt in range(max_retries):
            try:
                # If we make it through without error, break the loop
                response_text = llm.invoke(final_prompt).content
                # To be absolutely safe, we force a tiny 2-second sleep between every single question
                time.sleep(2) 
                break
            except Exception as e:
                error_msg = str(e)
                if "429" in error_msg or "rate_limit" in error_msg.lower():
                    wait_time = 20 # Wait 20 seconds for the Groq quota to reset
                    print(f"   [RATE LIMIT HIT] Sleeping for {wait_time} seconds... (Attempt {attempt+1}/{max_retries})")
                    time.sleep(wait_time)
                else:
                    raise e # If it's a different error, stop the program
        
        # Strict parsing using Regex
        match = re.search(r"<answer>\s*(\d+)\s*</answer>", response_text, re.IGNORECASE)
        if match:
            predicted_answer = match.group(1)
        else:
            # Fallback if it still disobeys
            digits = re.findall(r"\d", response_text)
            predicted_answer = digits[0] if digits else "None"
            
        print(f"Q{index + 1}: {question_text[:40]}...")
        print(f"LLM Output   : {response_text.strip()}")
        print(f"Cleaned Index: {predicted_answer}")
        print("-" * 30)
        
        evaluation_results.append({
            "Question": question_text,
            "Retrieved_Context": context_text,
            "LLM_Prediction": predicted_answer
        })
        
    print("\n5. Saving Full Evaluation Results...")
    results_df = pd.DataFrame(evaluation_results)
    output_path = "../data/benchmark/full_evaluation_results.csv"
    results_df.to_csv(output_path, index=False)
    print("DONE! You survived the API Rate Limits.")

except Exception as e:
    print(f"An error occurred: {e}")

1. Setting up API & LLM...
2. Connecting to the Massive Vector Database...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

3. Building the EXTREME Bulletproof XML Prompt...
4. Loading Benchmark Dataset (CSV)...
STARTING MASSIVE EVALUATION (RATE-LIMIT PROTECTED)
Q1: The quantum efficiency of a photon detec...
LLM Output   : <answer>2</answer>
Cleaned Index: 2
------------------------------
Q2: White light is normally incident on a pu...
LLM Output   : <answer>1</answer>
Cleaned Index: 1
------------------------------
Q3: Which of the following is true about any...
LLM Output   : <answer>2</answer>
Cleaned Index: 2
------------------------------
Q4: The best type of laser with which to do ...
LLM Output   : <answer>0</answer>
Cleaned Index: 0
------------------------------
Q5: Excited states of the helium atom can be...
LLM Output   : <answer>3</answer>
Cleaned Index: 3
------------------------------
Q6: Which of the following gives the total s...
LLM Output   : <answer>2</answer>
Cleaned Index: 2
------------------------------
Q7: Consider three identical, ideal capacito...
LLM Output   : <answer>1</answer>